# wattnet-client — quickstart with real data

This notebook exercises every method of `wattnet.client.WattnetClient` against the
**live, production** wattnet API (`https://api.wattnet.eu`) and keeps the real
responses as saved output, so you can see exactly what each call returns without
needing your own credentials just to read it.

To re-run this notebook yourself:

1. Get a bearer token (a pre-issued one, or via a credential's `client_id`/`client_secret` — see below).
2. Set it as an environment variable, **never hardcode it in the notebook**:

   ```bash
   export WATTNET_TOKEN="<your-token>"
   jupyter nbconvert --to notebook --execute --inplace examples/quickstart.ipynb
   ```

In [1]:
import os

from wattnet.client import WattnetClient, WattnetRawClient

client = WattnetClient(token=os.environ["WATTNET_TOKEN"])
client

WattnetClient(base_url='https://api.wattnet.eu/v1', auth=StaticTokenProvider)

Instead of a static token, you can also authenticate with the `client_id` and
`client_secret` of a wattnet credential — the client fetches and
caches a token for you (refreshing it automatically before it expires):

```python
client = WattnetClient(client_id="<your-client-id>", client_secret="<your-client-secret>")
```

We'll use a fixed 6-hour window for every time-series call below, so the examples
are reproducible and the tables stay a readable size.

In [2]:
START = "2026-07-15T00:00:00Z"
END = "2026-07-15T06:00:00Z"
ZONE = "ES"

## Zones

`get_zones()` — metadata and neighbours for every supported zone.

In [3]:
zones = client.get_zones()
print(zones.shape)
zones.head()

(60, 7)


,zone,full_name,eic_code,country_code,country_name,provider,neighbours
0,AL,Albania,10YAL-KESH-----5,ALB,Albania,ENTSO-E,"[GR, ME, RS, XK]"
1,AM,Armenia,10Y1001A1001B004,ARM,Armenia,ENTSO-E,[GE]
2,AT,Austria,10YAT-APG------L,AUT,Austria,ENTSO-E,"[CH, CZ, DE, HU, IT_NORTH, SI]"
3,AZ,Azerbaijan,10Y1001A1001B05V,AZE,Azerbaijan,ENTSO-E,[GE]
4,BA,Bosnia and Herzegovina,10YBA-JPCC-----D,BIH,Bosnia and Herzegovina,ENTSO-E,"[HR, ME, RS]"


## Energy metrics

### Generation

`get_generation()` — electricity generation by production type.

In [4]:
generation = client.get_generation(zone=ZONE, start=START, end=END)
print(generation.shape)
generation.head()

(384, 10)


,timestamp,zone,production_type,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-15 00:00:00+00:00,ES,biomass,516.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,biomass,516.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,biomass,516.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,biomass,512.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,biomass,512.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:54:28+00:00


Omit `zone` and the same call returns every supported zone in one request, rather than looping over zone codes yourself. The `zone` column tells the rows apart:

In [5]:
all_zones = client.get_generation(start=START, end=END)
print(all_zones.shape)
print(all_zones["zone"].nunique(), "zones in a single call")
all_zones.head()

(19968, 10)
52 zones in a single call


,timestamp,zone,production_type,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-15 00:00:00+00:00,AT,biomass,176.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,AT,biomass,176.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,AT,biomass,176.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,AT,biomass,176.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,AT,biomass,176.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:54:28+00:00


### Load

`get_load()` — total electricity demand.

In [6]:
load = client.get_load(zone=ZONE, start=START, end=END)
print(load.shape)
load.head()

(24, 9)


,timestamp,zone,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-15 00:00:00+00:00,ES,29032.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,28668.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,28436.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,28244.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,28080.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:54:28+00:00


Every zone-filtered method also accepts a `lat`/`lon` pair instead of a zone code — the API resolves the coordinate to the zone that contains it, which is handy when all you have is a site location:

In [7]:
by_coord = client.get_load(lat=40.4, lon=-3.7, start=START, end=END)
print(by_coord.shape)
by_coord.head()

(24, 9)


,timestamp,zone,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-15 00:00:00+00:00,ES,29032.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,28668.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,28436.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,28244.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,28080.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:54:28+00:00


Because `timestamp` arrives as a tz-aware `datetime64[ns, UTC]` column, standard pandas time-series tooling works with no conversion step — here, hourly means from the raw resolution:

In [8]:
hourly = load.set_index("timestamp").resample("1h").mean(numeric_only=True)
hourly

,value,valid
timestamp,,
2026-07-15 00:00:00+00:00,28595.0,1.0
2026-07-15 01:00:00+00:00,27690.0,1.0
2026-07-15 02:00:00+00:00,27216.0,1.0
2026-07-15 03:00:00+00:00,27455.0,1.0
2026-07-15 04:00:00+00:00,29623.0,1.0
2026-07-15 05:00:00+00:00,31712.0,1.0


Every row also carries `updated_at`, the moment that version of the value was stored. For long ranges the API can do the averaging itself: `resolution` (`"15m"`, `"1h"` or `"1d"`, buckets in UTC) returns far fewer rows:

In [9]:
daily = client.get_load(zone=ZONE, start="2026-07-01", end="2026-07-15", resolution="1d")
print(daily.shape)
daily.head()

(14, 9)


,timestamp,zone,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-01 00:00:00+00:00,ES,31986.12,MW,True,complete,official,ENTSO-E,2026-11-10 21:30:42+00:00
1,2026-07-02 00:00:00+00:00,ES,31406.54,MW,True,complete,official,ENTSO-E,2026-11-10 22:19:06+00:00
2,2026-07-03 00:00:00+00:00,ES,31217.62,MW,True,complete,official,ENTSO-E,2026-11-10 23:08:18+00:00
3,2026-07-04 00:00:00+00:00,ES,28019.21,MW,True,complete,official,ENTSO-E,2026-11-10 23:57:55+00:00
4,2026-07-05 00:00:00+00:00,ES,27409.79,MW,True,complete,official,ENTSO-E,2026-11-11 00:49:13+00:00


### Imports

`get_imports()` — electricity imported into the zone, by origin.

In [10]:
imports = client.get_imports(zone=ZONE, start=START, end=END)
print(imports.shape)
imports.head()

(48, 10)


,timestamp,zone,source,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-15 00:00:00+00:00,ES,FR,601.84,MW,True,complete,official,ENTSO-E,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,FR,646.92,MW,True,complete,official,ENTSO-E,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,FR,527.52,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,FR,522.84,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,FR,507.52,MW,True,complete,official,ENTSO-E,2026-11-11 09:54:28+00:00


### Exports

`get_exports()` — electricity exported from the zone, by destination.

In [11]:
exports = client.get_exports(zone=ZONE, start=START, end=END)
print(exports.shape)
exports.head()

(48, 10)


,timestamp,zone,destination,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-15 00:00:00+00:00,ES,FR,0.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,FR,0.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,FR,0.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,FR,0.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,FR,0.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:54:28+00:00


### Mix

`get_mix()` — flow-traced generation mix by production type.

In [12]:
mix = client.get_mix(zone=ZONE, start=START, end=END)
print(mix.shape)
mix.head()

(384, 10)


,timestamp,zone,production_type,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-15 00:00:00+00:00,ES,biomass,504.86,MW,True,complete,official,flow_tracing,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,biomass,502.14,MW,True,complete,official,flow_tracing,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,biomass,500.02,MW,True,complete,official,flow_tracing,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,biomass,495.33,MW,True,complete,official,flow_tracing,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,biomass,495.36,MW,True,complete,official,flow_tracing,2026-11-11 09:54:28+00:00


## Environmental metrics

### Footprints

`get_footprints()` — carbon/water footprint as a time series.

In [13]:
footprints = client.get_footprints(zone=ZONE, start=START, end=END)
print(footprints.shape)
footprints.head()

(48, 10)


,timestamp,zone,footprint_type,scope,value,unit,coverage,valid,zone_status,updated_at
0,2026-07-15 00:00:00+00:00,ES,carbon,life-cycle,231.17,gCO2/kWh,global,True,complete,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,carbon,life-cycle,231.46,gCO2/kWh,global,True,complete,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,carbon,life-cycle,234.46,gCO2/kWh,global,True,complete,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,carbon,life-cycle,233.86,gCO2/kWh,global,True,complete,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,carbon,life-cycle,237.09,gCO2/kWh,global,True,complete,2026-11-11 09:54:28+00:00


Set `aggregate=True` to get a single value per footprint type over the window instead of a time series:

In [14]:
footprints_agg = client.get_footprints(zone=ZONE, start=START, end=END, aggregate=True)
footprints_agg

,start,end,zone,footprint_type,scope,value,unit,coverage,valid,zone_status,aggregation_method,updated_at
0,2026-07-15T00:00:00Z,2026-07-15T06:00:00Z,ES,carbon,life-cycle,236.71,gCO2/kWh,global,True,complete,time-weighted-average,2026-11-11 10:07:16+00:00
1,2026-07-15T00:00:00Z,2026-07-15T06:00:00Z,ES,water,life-cycle,8.53,l/kWh,global,True,complete,time-weighted-average,2026-11-11 10:07:16+00:00


### Impacts

`get_impacts()` — water impact of electricity consumption.

In [15]:
impacts = client.get_impacts(zone=ZONE, start=START, end=END)
print(impacts.shape)
impacts.head()

(24, 10)


,timestamp,zone,impact_type,scope,value,unit,coverage,valid,zone_status,updated_at
0,2026-07-15 00:00:00+00:00,ES,water,operational,2.77,stress-l/kWh,global,True,complete,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,water,operational,2.65,stress-l/kWh,global,True,complete,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,water,operational,2.58,stress-l/kWh,global,True,complete,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,water,operational,2.58,stress-l/kWh,global,True,complete,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,water,operational,2.50,stress-l/kWh,global,True,complete,2026-11-11 09:54:28+00:00


### EnvironmentalScore

`get_environmental_score()` — a single 0-100 sustainability score.

In [16]:
environmental_score = client.get_environmental_score(zone=ZONE, start=START, end=END)
print(environmental_score.shape)
environmental_score.head()

(24, 8)


,timestamp,zone,scope,value,coverage,valid,zone_status,updated_at
0,2026-07-15 00:00:00+00:00,ES,operational,76.62,global,True,complete,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,operational,76.73,global,True,complete,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,operational,76.57,global,True,complete,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,operational,76.61,global,True,complete,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,operational,76.44,global,True,complete,2026-11-11 09:54:28+00:00


## Shares metrics

### Flow share

`get_flow_share()` — share of this zone's production exported to each destination zone.

In [17]:
flow_share = client.get_flow_share(zone=ZONE, start=START, end=END)
print(flow_share.shape)
flow_share.head()

(1248, 8)


,timestamp,zone,destination,value,unit,valid,zone_status,updated_at
0,2026-07-15 00:00:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:54:28+00:00


### Mix share

`get_mix_share()` — share of this zone's mix coming from each origin zone.

In [18]:
mix_share = client.get_mix_share(zone=ZONE, start=START, end=END)
print(mix_share.shape)
mix_share.head()

(1248, 8)


,timestamp,zone,origin,value,unit,valid,zone_status,updated_at
0,2026-07-15 00:00:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,AT,0.0,%,True,complete,2026-11-11 09:54:28+00:00


### Footprint share

`get_footprint_share()` — this zone's footprint decomposed by origin zone.

In [19]:
footprint_share = client.get_footprint_share(zone=ZONE, start=START, end=END)
print(footprint_share.shape)
footprint_share.head()

(2496, 10)


,timestamp,zone,source,footprint_type,scope,value,unit,valid,zone_status,updated_at
0,2026-07-15 00:00:00+00:00,ES,AT,carbon,life-cycle,0.0,gCO2/kWh,True,complete,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,AT,carbon,life-cycle,0.0,gCO2/kWh,True,complete,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,AT,carbon,life-cycle,0.0,gCO2/kWh,True,complete,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,AT,carbon,life-cycle,0.0,gCO2/kWh,True,complete,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,AT,carbon,life-cycle,0.0,gCO2/kWh,True,complete,2026-11-11 09:54:28+00:00


### Impact share

`get_impact_share()` — this zone's impact decomposed by origin zone.

In [20]:
impact_share = client.get_impact_share(zone=ZONE, start=START, end=END)
print(impact_share.shape)
impact_share.head()

(1248, 10)


,timestamp,zone,source,impact_type,scope,value,unit,valid,zone_status,updated_at
0,2026-07-15 00:00:00+00:00,ES,AT,water,operational,0.0,stress-l/kWh,True,complete,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,AT,water,operational,0.0,stress-l/kWh,True,complete,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,AT,water,operational,0.0,stress-l/kWh,True,complete,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,AT,water,operational,0.0,stress-l/kWh,True,complete,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,AT,water,operational,0.0,stress-l/kWh,True,complete,2026-11-11 09:54:28+00:00


## Factors

`get_factors()` — global emission/consumption factors, independent of any zone.

In [21]:
factors = client.get_factors(
    factor_type="carbon", production_type="solar", start=START, end=END
)
print(factors.shape)
factors.head()

(48, 10)


,timestamp,production_type,factor_type,scope,value,unit,source,year,source_link,updated_at
0,2026-07-15 00:00:00+00:00,solar,carbon,life-cycle,36.95,gCO2/kWh,UNECE,2022,https://unece.org/sites/default/files/2022-04/...,2026-11-11 09:47:42+00:00
1,2026-07-15 00:15:00+00:00,solar,carbon,life-cycle,36.95,gCO2/kWh,UNECE,2022,https://unece.org/sites/default/files/2022-04/...,2026-11-11 09:48:21+00:00
2,2026-07-15 00:30:00+00:00,solar,carbon,life-cycle,36.95,gCO2/kWh,UNECE,2022,https://unece.org/sites/default/files/2022-04/...,2026-11-11 09:49:02+00:00
3,2026-07-15 00:45:00+00:00,solar,carbon,life-cycle,36.95,gCO2/kWh,UNECE,2022,https://unece.org/sites/default/files/2022-04/...,2026-11-11 09:49:43+00:00
4,2026-07-15 01:00:00+00:00,solar,carbon,life-cycle,36.95,gCO2/kWh,UNECE,2022,https://unece.org/sites/default/files/2022-04/...,2026-11-11 09:50:23+00:00


## Status

`get_status()` and its per-dependency variants report the health of the API and its upstream providers.

In [22]:
status = client.get_status()
status

,storage,entso-e,elexon,epias
0,up,up,up,up


## Raw JSON access

`WattnetRawClient` exposes the exact same methods but returns the untouched JSON payload instead of a DataFrame — useful if you want to handle the nested structure yourself.

In [23]:
raw_client = WattnetRawClient(token=os.environ["WATTNET_TOKEN"])
zones_json = raw_client.get_zones()
print(type(zones_json), len(zones_json))
zones_json[0]

<class 'list'> 60


{'zone': 'AL',
 'full_name': 'Albania',
 'eic_code': '10YAL-KESH-----5',
 'country_code': 'ALB',
 'country_name': 'Albania',
 'provider': 'ENTSO-E',
 'neighbours': ['GR', 'ME', 'RS', 'XK']}

## Error handling

All errors derive from `wattnet.client.WattnetError`. Misconfiguring the client (e.g. no auth at all) raises `ConfigurationError` immediately, without making any request:

In [24]:
from wattnet.client import ConfigurationError

try:
    WattnetClient()
except ConfigurationError as exc:
    print(f"ConfigurationError: {exc}")

ConfigurationError: Must provide either `token`, or both `client_id` and `client_secret`.


Errors coming back from the API itself raise `WattnetAPIError`, which carries the HTTP `status_code`, the API's own `detail` message and a stable `code` so you can branch on them instead of parsing strings. Here, a coordinate in the middle of the Atlantic resolves to no zone at all:

In [25]:
from wattnet.client import WattnetAPIError

try:
    client.get_load(lat=0.0, lon=-30.0, start=START, end=END)
except WattnetAPIError as exc:
    print(f"status_code: {exc.status_code}")
    print(f"detail:      {exc.detail}")
    print(f"code:        {exc.code}")

status_code: 404
detail:      No zone found for the provided coordinates.
code:        zone_not_found_for_coordinates


Note that an *unrecognised zone code* is **not** an error: the API does not validate zone codes against its zone list, it just filters on them, so you get a successful response with zero rows. Check `.empty` rather than relying on an exception:

In [26]:
missing = client.get_generation(zone="NOT-A-ZONE", start=START, end=END)
print(f"empty: {missing.empty}  shape: {missing.shape}")

empty: True  shape: (0, 0)


## Async clients

For asyncio applications there are `AsyncWattnetClient` (DataFrames) and `AsyncWattnetRawClient` (raw JSON). They expose the same methods as their synchronous counterparts, every one of them a coroutine, and are used as async context managers so the underlying `aiohttp` session is closed for you. Jupyter supports top-level `await`, so we can call them directly:

In [27]:
from wattnet.client import AsyncWattnetClient

async with AsyncWattnetClient(token=os.environ["WATTNET_TOKEN"]) as async_client:
    async_generation = await async_client.get_generation(zone=ZONE, start=START, end=END)
print(async_generation.shape)
async_generation.head()

(384, 10)


,timestamp,zone,production_type,value,unit,valid,zone_status,data_state,datasource,updated_at
0,2026-07-15 00:00:00+00:00,ES,biomass,516.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:51:44+00:00
1,2026-07-15 00:15:00+00:00,ES,biomass,516.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:52:26+00:00
2,2026-07-15 00:30:00+00:00,ES,biomass,516.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:08+00:00
3,2026-07-15 00:45:00+00:00,ES,biomass,512.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:53:46+00:00
4,2026-07-15 01:00:00+00:00,ES,biomass,512.0,MW,True,complete,official,ENTSO-E,2026-11-11 09:54:28+00:00


Because the calls are coroutines, independent requests can run concurrently with `asyncio.gather`:

In [28]:
import asyncio

from wattnet.client import AsyncWattnetRawClient

async with AsyncWattnetRawClient(token=os.environ["WATTNET_TOKEN"]) as async_raw:
    load, generation = await asyncio.gather(
        async_raw.get_load(zone=ZONE, start=START, end=END),
        async_raw.get_generation(zone=ZONE, start=START, end=END),
    )
print(f"load rows: {len(load)}  generation rows: {len(generation)}")

load rows: 1  generation rows: 1
